In [23]:
import pandas as pd
import numpy as np

In [24]:
reading_df = pd.read_csv("Dataset/reading_scores.csv")   # contains ELA + Math
science_df = pd.read_csv("Dataset/science_scores.csv")   # contains Science

/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_86384/1636155830.py:1: DtypeWarning: Columns (93,128,168,333,410,533,568,608,773,850) have mixed types. Specify dtype option on import or set low_memory=False.
  reading_df = pd.read_csv("Dataset/reading_scores.csv")   # contains ELA + Math
/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_86384/1636155830.py:2: DtypeWarning: Columns (88,108,230,250) have mixed types. Specify dtype option on import or set low_memory=False.
  science_df = pd.read_csv("Dataset/science_scores.csv")   # contains Science


In [25]:
# selecting only the elementary schools 
reading_df = reading_df[reading_df["School Type"] == "Elementary School"].copy()
science_df = science_df[science_df["School Type"] == "Elementary School"].copy()

In [26]:
def clean_district(df):
    df["District"] = (
        df["District"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df

reading_df = clean_district(reading_df)
science_df = clean_district(science_df)


In [27]:
# choosing predictors from the reading & math files 
reading_df = reading_df[
    [
        "District",
        "% ELA Proficiency",
        "% ELA Proficiency - Low Income",
        "% ELA Proficiency - Children with Disabilities",
        "% Math Proficiency",
        "% Math Proficiency - Low Income",
        "% Math Proficiency - Children with Disabilities"
    ]
]



In [28]:
# choosing predictors from the science scores  
science_df = science_df[
    [
        "District",
        "% Science Proficiency",
        "% Science Proficiency - Low Income",
        "% Science Proficiency - Children with Disabilities"
    ]
]



In [29]:
# converting the scores to numeric data 
for df in [reading_df, science_df]:
    for col in df.columns:
        if col != "District":
            df[col] = pd.to_numeric(df[col], errors="coerce")



In [30]:
# grouping the score by district and averaging 
reading_district = (
    reading_df
    .groupby("District", as_index=False)
    .mean()
)

science_district = (
    science_df
    .groupby("District", as_index=False)
    .mean()
)



In [31]:
#renaming the columns 
reading_district = reading_district.rename(columns={
    "% ELA Proficiency": "ELA_Proficiency",
    "% ELA Proficiency - Low Income": "ELA_Proficiency_LowIncome",
    "% ELA Proficiency - Children with Disabilities": "ELA_Proficiency_CWD",
    "% Math Proficiency": "Math_Proficiency",
    "% Math Proficiency - Low Income": "Math_Proficiency_LowIncome",
    "% Math Proficiency - Children with Disabilities": "Math_Proficiency_CWD"
})

science_district = science_district.rename(columns={
    "% Science Proficiency": "Science_Proficiency",
    "% Science Proficiency - Low Income": "Science_Proficiency_LowIncome",
    "% Science Proficiency - Children with Disabilities": "Science_Proficiency_CWD"
})


In [32]:
# merging the datasets 
academic_scores = reading_district.merge(
    science_district,
    on="District",
    how="inner"
)


In [33]:
print("Final dataset shape:", academic_scores.shape)
print(academic_scores.head())

academic_scores.to_csv(
    "academic_scores_elementary_district.csv",
    index=False
)


Final dataset shape: (762, 10)
                     District  ELA_Proficiency  ELA_Proficiency_LowIncome  \
0        A-C CENTRAL CUSD 262        56.500000                  50.000000   
1      ABINGDON-AVON CUSD 276        52.000000                  38.600000   
2  ACE AMANDLA CHARTER SCHOOL              NaN                        NaN   
3                ADDISON SD 4        40.028571                  34.428571   
4                AKIN CCSD 91        68.900000                  60.000000   

   ELA_Proficiency_CWD  Math_Proficiency  Math_Proficiency_LowIncome  \
0                  NaN         43.500000                         NaN   
1                  NaN         45.800000                   29.700000   
2                  NaN               NaN                         NaN   
3                 25.0         30.214286                   25.657143   
4                  NaN         57.400000                   48.600000   

   Math_Proficiency_CWD  Science_Proficiency  Science_Proficiency_LowInco

In [36]:
academic_df = pd.read_csv("Dataset/district_scores_imputed_MAR.csv")
meals_df = pd.read_csv("Dataset/meals.csv")

In [37]:
def clean_district(df, col):
    df[col] = (
        df[col]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df

academic_df = clean_district(academic_df, "District")
meals_df = clean_district(meals_df, "Districts")


In [38]:
meals_numeric_cols = meals_df.select_dtypes(include="number").columns.tolist()


In [39]:
print(meals_numeric_cols)

['ROE', 'student_disabilities_percent', 'student_low_income_percent', 'days_entered.x', 'HEI 2015 Total Score', 'HEI 2015 Total Fruits (0-5)', 'HEI 2015 Whole Fruits (0-5)', 'HEI 2015 Total Vegetables (0-5)', 'HEI 2015 Greens and Beans (0-5)', 'HEI 2015 Whole Grains (0-10)', 'HEI 2015 Dairy (0-10)', 'HEI 2015 Total Protein Foods (0-5)', 'HEI 2015 Seafood and Plant Proteins (0-5)', 'HEI 2015 Fatty Acids (0-10)', 'HEI 2015 Refined Grains (0-10)', 'HEI 2015 Sodium (0-10)', 'HEI 2015 Added Sugars (0-10)', 'HEI 2015 Saturated Fats (0-10)', 'Total Fruit Servings in cup equivalents per 1000 kcal', 'Whole Fruit Servings in cup equivalents per 1000 kcal', 'Total Vegetable Servings in cup equivalents per 1000 kcal', 'Greens and Beans Servings in cup equivalents per 1000 kcal', 'Whole Grain Servings in ounce equivalents per 1000 kcal', 'Dairy Servings in cup equivalents per 1000 kcal', 'Total Protein Servings in ounce equivalents per 1000 kcal', 'Seafood and Plant Protein Servings in ounce equiva

In [40]:
meals_district = (
    meals_df
    .groupby("Districts", as_index=False)[meals_numeric_cols]
    .mean()
)

In [41]:
meals_district = meals_district.rename(columns={
    "Districts": "District"
})

In [42]:
final_df = meals_district.merge(
    academic_df,
    on="District",
    how="left"
)


In [43]:
print("Final dataset shape:", final_df.shape)
print(final_df.head())

Final dataset shape: (54, 263)
                         District   ROE  student_disabilities_percent  \
0             AURORA EAST USD 131  31.0                          16.0   
1             BALL CHATHAM CUSD 5  51.0                          19.0   
2               BEECHER CUSD 200U  56.0                          21.3   
3              BRIMFIELD CUSD 309  48.0                          12.4   
4  CARRIER MILLS-STONEFORT CUSD 2  20.0                          23.2   

   student_low_income_percent  days_entered.x  HEI 2015 Total Score  \
0                        70.0            20.0             59.977550   
1                        23.9            20.0             51.193643   
2                        21.9            20.0             43.690050   
3                        16.8            20.0             60.763450   
4                        88.3            14.0             64.624500   

   HEI 2015 Total Fruits (0-5)  HEI 2015 Whole Fruits (0-5)  \
0                     4.715150          

In [44]:
final_df.to_csv(
    "academic_meals_elementary_district.csv",
    index=False
)

In [45]:
import os
os.getcwd()


'/Users/ishita/Coder/Proj/STUDENT-MEALS-'